In [1]:
import shutil

!mkdir -p /kaggle/working/animal_dataset
shutil.copytree("/kaggle/input/c2342342/Data/Kangaroo-Koala.v14-kangaroo-koala-v13.yolov8", 
                "/kaggle/working/animal_dataset", dirs_exist_ok=True)

'/kaggle/working/animal_dataset'

In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.1 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 31.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.3 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 93.4 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.9.41
    Uninstalling nvidia-nvjitlink-cu12-12.9.41:
      Successfully uninstalled nvidia-nvjitlink-cu12-12.9.41
  Attempting uninstall: nvidia-curand-cu12
    Found existing ins

In [3]:
import os, cv2, shutil
import albumentations as A

# Paths
image_dir = "/kaggle/working/animal_dataset/train/images"
label_dir = "/kaggle/working/animal_dataset/train/labels"
aug_img_dir = "/kaggle/working/animal_dataset/train/images_aug"
aug_lbl_dir = "/kaggle/working/animal_dataset/train/labels_aug"

os.makedirs(aug_img_dir, exist_ok=True)
os.makedirs(aug_lbl_dir, exist_ok=True)

# Albumentations transform
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.Blur(blur_limit=3, p=0.2),
    A.RandomGamma(p=0.3)
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

def is_valid_bbox(x, y, w, h):
    return (w > 0.01 and h > 0.01 and
            x - w/2 >= 0 and x + w/2 <= 1 and
            y - h/2 >= 0 and y + h/2 <= 1)

def augment_image(image_path, label_path, save_id):
    img = cv2.imread(image_path)
    h, w = img.shape[:2]
    bboxes, class_labels = [], []

    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            cls = int(parts[0])
            x, y, bw, bh = map(float, parts[1:])
            if is_valid_bbox(x, y, bw, bh):
                bboxes.append([x, y, bw, bh])
                class_labels.append(str(cls))

    if not bboxes: return  # bỏ nếu không có bbox hợp lệ

    try:
        transformed = transform(image=img, bboxes=bboxes, class_labels=class_labels)
    except Exception:
        return  # bỏ nếu augmentation làm bbox lỗi

    if not transformed['bboxes']: return  # bỏ nếu bbox bị xóa hết

    out_img = os.path.join(aug_img_dir, f"aug_{save_id}.jpg")
    out_lbl = os.path.join(aug_lbl_dir, f"aug_{save_id}.txt")
    cv2.imwrite(out_img, transformed['image'])

    with open(out_lbl, 'w') as f:
        for cls, box in zip(transformed['class_labels'], transformed['bboxes']):
            f.write(f"{cls} {box[0]:.6f} {box[1]:.6f} {box[2]:.6f} {box[3]:.6f}\n")

save_id = 0
for fname in os.listdir(label_dir):
    if not fname.endswith(".txt"): continue
    img_path = os.path.join(image_dir, fname.replace(".txt", ".jpg"))
    lbl_path = os.path.join(label_dir, fname)
    if not os.path.exists(img_path): continue
    for _ in range(3):  # mỗi ảnh augment 3 lần
        augment_image(img_path, lbl_path, save_id)
        save_id += 1

print(f" Đã tạo {save_id} ảnh augment hợp lệ.")

/usr/local/lib/python3.11/dist-packages/albumentations/__init__.py:28: UserWarning: A new version of Albumentations is available: '2.0.8' (you have '2.0.5'). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


 Đã tạo 5574 ảnh augment hợp lệ.


In [4]:
def merge_augmented_data(aug_img_dir, aug_lbl_dir, main_img_dir, main_lbl_dir):
    img_count = 0
    for f in os.listdir(aug_img_dir):
        shutil.copy(os.path.join(aug_img_dir, f), os.path.join(main_img_dir, f))
        img_count += 1
    for f in os.listdir(aug_lbl_dir):
        shutil.copy(os.path.join(aug_lbl_dir, f), os.path.join(main_lbl_dir, f))
    print(f" Đã merge {img_count} ảnh augment vào train.")

# Gọi merge
merge_augmented_data(aug_img_dir, aug_lbl_dir, image_dir, label_dir)

 Đã merge 5403 ảnh augment vào train.


In [5]:
import os
import cv2
import numpy as np
import random
import shutil

# === Đường dẫn gốc ===
base_dir = "/kaggle/working/animal_dataset/train"
image_dir = os.path.join(base_dir, "images")
label_dir = os.path.join(base_dir, "labels")

mosaic_out = os.path.join(base_dir, "mosaic")
cutmix_out = os.path.join(base_dir, "cutmix")
os.makedirs(mosaic_out + "/images", exist_ok=True)
os.makedirs(mosaic_out + "/labels", exist_ok=True)
os.makedirs(cutmix_out + "/images", exist_ok=True)
os.makedirs(cutmix_out + "/labels", exist_ok=True)


def load_yolo_label(label_path, img_w, img_h):
    boxes, classes = [], []
    with open(label_path, 'r') as f:
        for line in f:
            cls, x, y, w, h = map(float, line.strip().split())
            if w <= 0 or h <= 0: continue
            x1 = (x - w / 2) * img_w
            y1 = (y - h / 2) * img_h
            x2 = (x + w / 2) * img_w
            y2 = (y + h / 2) * img_h
            boxes.append([x1, y1, x2, y2])
            classes.append(int(cls))
    return boxes, classes


def save_yolo_labels(path, boxes, classes, img_w, img_h):
    with open(path, 'w') as f:
        for box, cls in zip(boxes, classes):
            x1, y1, x2, y2 = box
            x = ((x1 + x2) / 2) / img_w
            y = ((y1 + y2) / 2) / img_h
            w = (x2 - x1) / img_w
            h = (y2 - y1) / img_h
            if w <= 0 or h <= 0 or x < 0 or x > 1 or y < 0 or y > 1:
                continue
            f.write(f"{cls} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n")


def mosaic_augment(n_samples=10, out_size=640):
    image_files = [f for f in os.listdir(image_dir) if f.endswith('.jpg')]
    for i in range(n_samples):
        selected = random.sample(image_files, 4)
        mosaic_img = np.full((out_size * 2, out_size * 2, 3), 114, dtype=np.uint8)
        labels, classes = [], []

        for idx, fname in enumerate(selected):
            img_path = os.path.join(image_dir, fname)
            lbl_path = os.path.join(label_dir, fname.replace(".jpg", ".txt"))
            img = cv2.imread(img_path)
            h, w = img.shape[:2]
            boxes, clss = load_yolo_label(lbl_path, w, h)
            img = cv2.resize(img, (out_size, out_size))
            x_off = (idx % 2) * out_size
            y_off = (idx // 2) * out_size
            mosaic_img[y_off:y_off+out_size, x_off:x_off+out_size] = img

            for box, cls in zip(boxes, clss):
                x1, y1, x2, y2 = box
                x1 = x1 * out_size / w + x_off
                y1 = y1 * out_size / h + y_off
                x2 = x2 * out_size / w + x_off
                y2 = y2 * out_size / h + y_off
                labels.append([x1, y1, x2, y2])
                classes.append(cls)

        out_img = os.path.join(mosaic_out, "images", f"mosaic_{i}.jpg")
        out_lbl = os.path.join(mosaic_out, "labels", f"mosaic_{i}.txt")
        cv2.imwrite(out_img, mosaic_img)
        save_yolo_labels(out_lbl, labels, classes, out_size * 2, out_size * 2)


def cutmix_augment(n_samples=10, out_size=640):
    image_files = [f for f in os.listdir(image_dir) if f.endswith('.jpg')]
    for i in range(n_samples):
        a, b = random.sample(image_files, 2)
        img_a = cv2.imread(os.path.join(image_dir, a))
        img_b = cv2.imread(os.path.join(image_dir, b))
        img_a = cv2.resize(img_a, (out_size, out_size))
        img_b = cv2.resize(img_b, (out_size, out_size))

        cut_x = random.randint(out_size//4, 3*out_size//4)
        cut_y = random.randint(out_size//4, 3*out_size//4)
        img_a[cut_y:, cut_x:] = img_b[cut_y:, cut_x:]

        labels, classes = [], []
        for fname, offset_x, offset_y in [(a, 0, 0), (b, cut_x, cut_y)]:
            lbl_path = os.path.join(label_dir, fname.replace(".jpg", ".txt"))
            boxes, clss = load_yolo_label(lbl_path, out_size, out_size)
            for box, cls in zip(boxes, clss):
                x1, y1, x2, y2 = box
                cx = (x1 + x2) / 2
                cy = (y1 + y2) / 2
                if fname == b and (cx < cut_x or cy < cut_y):
                    continue
                labels.append([x1, y1, x2, y2])
                classes.append(cls)

        out_img = os.path.join(cutmix_out, "images", f"cutmix_{i}.jpg")
        out_lbl = os.path.join(cutmix_out, "labels", f"cutmix_{i}.txt")
        cv2.imwrite(out_img, img_a)
        save_yolo_labels(out_lbl, labels, classes, out_size, out_size)


def merge_augmented(aug_dir):
    for f in os.listdir(os.path.join(aug_dir, "images")):
        shutil.copy(os.path.join(aug_dir, "images", f), os.path.join(image_dir, f))
    for f in os.listdir(os.path.join(aug_dir, "labels")):
        shutil.copy(os.path.join(aug_dir, "labels", f), os.path.join(label_dir, f))

# === Gọi augment ===
mosaic_augment(n_samples=20)
cutmix_augment(n_samples=20)

# === Gộp kết quả ===
merge_augmented(mosaic_out)
merge_augmented(cutmix_out)
print(" Mosaic + CutMix + Merge.")


 Mosaic + CutMix + Merge.


In [6]:
from ultralytics import YOLO

model = YOLO("yolov8x.pt")

# Train
model.train(
    data="/kaggle/working/animal_dataset/data.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    augment=True,
    workers=2,
    name="animal_aug_train",
    amp=True           # Mixed precision
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


100%|██████████| 131M/131M [00:03<00:00, 45.0MB/s] 


Ultralytics 8.3.147 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/animal_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8x.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=animal_aug_train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0,

100%|██████████| 755k/755k [00:00<00:00, 130MB/s]


Overriding model.yaml nc=80 with nc=4

                   from  n    params  module                                       arguments                     
  0                  -1  1      2320  ultralytics.nn.modules.conv.Conv             [3, 80, 3, 2]                 
  1                  -1  1    115520  ultralytics.nn.modules.conv.Conv             [80, 160, 3, 2]               
  2                  -1  3    436800  ultralytics.nn.modules.block.C2f             [160, 160, 3, True]           
  3                  -1  1    461440  ultralytics.nn.modules.conv.Conv             [160, 320, 3, 2]              
  4                  -1  6   3281920  ultralytics.nn.modules.block.C2f             [320, 320, 6, True]           
  5                  -1  1   1844480  ultralytics.nn.modules.conv.Conv             [320, 640, 3, 2]              
  6                  -1  6  13117440  ultralytics.nn.modules.block.C2f             [640, 640, 6, True]           
  7                  -1  1   3687680  ultralytics

...

                   all        529        669      0.868      0.844      0.894      0.625



30 epochs completed in 5.241 hours.
Optimizer stripped from runs/detect/animal_aug_train/weights/last.pt, 136.7MB
Optimizer stripped from runs/detect/animal_aug_train/weights/best.pt, 136.7MB

Validating runs/detect/animal_aug_train/weights/best.pt...
Ultralytics 8.3.147 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 112 layers, 68,127,420 parameters, 0 gradients, 257.4 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 17/17 [00:25<00:00,  1.48s/it]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        529        669      0.854      0.843      0.883      0.628
                     0        265        286      0.761      0.734      0.787       0.47
                     1        256        383      0.948      0.952      0.979      0.786
Speed: 0.1ms preprocess, 44.1ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to runs/detect/animal_aug_train


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7d3d884b8910>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

In [7]:
metrics = model.val()
print(metrics.box)  # Hiển thị các chỉ số như mAP50, mAP50-95, Precision, Recall

Ultralytics 8.3.147 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 112 layers, 68,127,420 parameters, 0 gradients, 257.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1384.7±321.9 MB/s, size: 47.2 KB)


val: Scanning /kaggle/working/animal_dataset/valid/labels.cache... 529 images, 11 backgrounds, 0 corrupt: 100%|██████████| 529/529 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 34/34 [00:25<00:00,  1.31it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        529        669      0.854      0.843      0.883      0.629
                     0        265        286      0.761      0.734      0.788      0.472
                     1        256        383      0.948      0.952      0.979      0.786
Speed: 0.6ms preprocess, 44.9ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to runs/detect/animal_aug_train2
ultralytics.utils.metrics.Metric object with attributes:

all_ap: array([[    0.78775,     0.76612,     0.74736,     0.67278,     0.57809,     0.50382,      0.3388,     0.22215,    0.090361,    0.012483],
       [    0.97918,     0.97703,     0.96622,     0.94013,     0.90906,     0.87267,     0.80246,     0.69881,     0.53679,     0.17833]])
ap: array([    0.47197,     0.78607])
ap50: array([    0.78775,     0.97918])
ap_class_index: array([0, 1])
curves: []
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.00800

In [8]:
metrics = model.val(
    data="/kaggle/working/animal_dataset/data.yaml",
    split="test",  # nếu bạn có tập test được định nghĩa
    imgsz=640
)

# Truy cập vào object con: metrics.box
precision = metrics.box.p.mean()     # Mean of Precision 
recall = metrics.box.r.mean()        # Mean of Recall
map50 = metrics.box.map50            # mAP@0.5
map_5095 = metrics.box.map           # mAP@0.5:0.95

# F1-score
f1_score = 2 * (precision * recall) / (precision + recall + 1e-6)

# In kết quả
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1_score:.4f}")
print(f"mAP@0.5: {map50:.4f}")
print(f"mAP@0.5:0.95: {map_5095:.4f}")

Ultralytics 8.3.147 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 20.1±3.0 MB/s, size: 39.5 KB)


val: Scanning /kaggle/working/animal_dataset/test/labels... 267 images, 4 backgrounds, 0 corrupt: 100%|██████████| 267/267 [00:00<00:00, 735.58it/s]

val: New cache created: /kaggle/working/animal_dataset/test/labels.cache



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 17/17 [00:13<00:00,  1.25it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        267        375      0.834      0.793      0.852      0.568
                     0        144        164      0.741      0.689      0.753      0.447
                     1        126        211      0.927      0.897      0.952       0.69
Speed: 1.0ms preprocess, 45.2ms inference, 0.0ms loss, 1.4ms postprocess per image
Results saved to runs/detect/animal_aug_train3
Precision: 0.8336
Recall: 0.7931
F1-score: 0.8128
mAP@0.5: 0.8524
mAP@0.5:0.95: 0.5684


In [9]:
import cv2
import numpy as np
from ultralytics import YOLO
from collections import deque
import pandas as pd

# ======== Định nghĩa các lớp phụ trợ ========
class SpeciesRiskConfig:
    def __init__(self, risk_dict=None):
        self.risk_dict = risk_dict or {'kangaroo': 90, 'koala': 10}
    def score(self, name: str) -> int:
        return self.risk_dict.get(name.lower(), 0)

class RoadAreaChecker:
    def __init__(self, polygon_points, img_w, img_h):
        self.mask = self._poly_to_mask(polygon_points, img_w, img_h)
    def _poly_to_mask(self, pts, w, h):
        mask = np.zeros((h, w), dtype=np.uint8)
        cv2.fillPoly(mask, [np.array(pts, dtype=np.int32)], 1)
        return mask.astype(bool)
    def is_near_road(self, bbox):
        x1, y1, x2, y2 = map(int, bbox)
        x1, y1 = max(x1, 0), max(y1, 0)
        x2, y2 = min(x2, self.mask.shape[1]-1), min(y2, self.mask.shape[0]-1)
        roi = self.mask[y1:y2+1, x1:x2+1]
        return roi.any()

class DistanceEstimator:
    def __init__(self, img_h, near_ratio=0.4):
        self.img_h = img_h
        self.near_ratio = near_ratio
    def is_close(self, bbox):
        x1, y1, x2, y2 = bbox
        box_h = y2 - y1
        return (box_h / self.img_h) >= self.near_ratio

class DayNightDetector:
    def __init__(self, threshold=60):
        self.threshold = threshold
    def get_time_of_day(self, img_bgr):
        y_cr_cb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2YCrCb)
        y_mean  = y_cr_cb[..., 0].mean()
        return 'night' if y_mean < self.threshold else 'day'

class RiskSmoother:
    def __init__(self, window=5, hi_thresh=0.6):
        self.window = window
        self.hi_thresh = hi_thresh
        self.buffer = deque(maxlen=window)
    def update(self, risk_lvl: str) -> str:
        self.buffer.append(risk_lvl)
        if len(self.buffer) < self.window:
            return risk_lvl
        high_ratio = sum(r=='HIGH' for r in self.buffer)/self.window
        return 'HIGH' if high_ratio >= self.hi_thresh else self.buffer[-1]

# ======== Lớp chính: ThreatAssessmentAgent V2 ========
class ThreatAssessmentAgentV2:
    def __init__(self,
                 species_risk: SpeciesRiskConfig,
                 road_checker : RoadAreaChecker,
                 dist_est     : DistanceEstimator,
                 day_night    : DayNightDetector,
                 smoother     : RiskSmoother,
                 high_score=70,
                 med_score=40,
                 conf_thr=0.6):
        self.species_risk  = species_risk
        self.road_checker  = road_checker
        self.dist_est      = dist_est
        self.day_night_det = day_night
        self.smoother      = smoother
        self.high_score    = high_score
        self.med_score     = med_score
        self.conf_thr      = conf_thr

    def assess_risk(self, img_bgr, det):
        cls   = det['class_name'].lower()
        conf  = det['confidence']
        bbox  = det['bbox']
        tod = det.get('time_of_day') or self.day_night_det.get_time_of_day(img_bgr)
        score = self.species_risk.score(cls)
        near_road = self.road_checker.is_near_road(bbox)
        close_obj = self.dist_est.is_close(bbox)

        risk = 'LOW'
        if score >= self.high_score and (near_road or close_obj):
            risk = 'HIGH'
        elif (score >= self.med_score and tod == 'night' and conf >= self.conf_thr):
            risk = 'MEDIUM'
        return self.smoother.update(risk)

# ======== Cấu hình đường dẫn & mô hình YOLO ========
model_path = "/kaggle/working/runs/detect/animal_aug_train/weights/best.pt"
image_path = "/kaggle/working/animal_dataset/test/images/00001_jpg.rf.a47ad67a15d9815ec15eaa4ebbb936ef.jpg"

img = cv2.imread(image_path)
H, W = img.shape[:2]
model = YOLO(model_path)
results = model(img)[0]

# ======== Cấu hình agent ========
species_cfg = SpeciesRiskConfig({'kangaroo': 90, 'koala': 10})
road_poly   = [(0, H-130), (W, H-130), (W, H), (0, H)]
road_chk    = RoadAreaChecker(road_poly, img_w=W, img_h=H)
dist_est    = DistanceEstimator(img_h=H, near_ratio=0.45)
daynight    = DayNightDetector(threshold=70)
smoother    = RiskSmoother(window=5, hi_thresh=0.6)

agent = ThreatAssessmentAgentV2(
    species_risk=species_cfg,
    road_checker=road_chk,
    dist_est=dist_est,
    day_night=daynight,
    smoother=smoother,
    high_score=70,
    med_score=40,
    conf_thr=0.6
)

# ======== Đánh giá rủi ro từ YOLO output ========
outputs = []
for r in results.boxes:
    det = {
        'class_name' : model.names[int(r.cls)],
        'confidence' : float(r.conf),
        'bbox': r.xyxy[0].tolist(),  # Lấy hàng đầu tiên rồi chuyển thành list
    }
    risk = agent.assess_risk(img, det)
    outputs.append({
        'class'     : det['class_name'],
        'confidence': det['confidence'],
        'bbox'      : det['bbox'],
        'risk'      : risk
    })

# ======== Hiển thị kết quả ========
df = pd.DataFrame(outputs)
display(df)


0: 448x640 2 1s, 42.3ms
Speed: 2.7ms preprocess, 42.3ms inference, 1.7ms postprocess per image at shape (1, 3, 448, 640)


,class,confidence,bbox,risk
0,1,0.938376,"[175.9144744873047, 131.48611450195312, 343.53...",LOW
1,1,0.933810,"[311.0578308105469, 115.64299774169922, 448.76...",LOW


In [11]:
class CommunicationAgent:
    def __init__(self, enable_mobile_alert=False):
    
        self.enable_mobile_alert = enable_mobile_alert

    def generate_message(self, risk_level):
       
        if risk_level == "HIGH":
            return " [CRITICAL] HIGH risk detected! Immediate action required."
        elif risk_level == "MEDIUM":
            return " [WARNING] Medium risk detected. Monitor the situation."
        else:
            return " [SAFE] No significant threat detected."

    def send_alert(self, message):
      
        print(" [ROAD DISPLAY SYSTEM]")
        print(message)
        if self.enable_mobile_alert:
            print(" [MOBILE APP NOTIFICATION]")
            print(message)

    def handle_risk(self, risk_level):
   
        if risk_level in ["HIGH", "MEDIUM"]:
            msg = self.generate_message(risk_level)
            self.send_alert(msg)
        else:
            print(" [STATUS] LOW risk - no alert needed.")
risk_level = "HIGH"  # hoặc "MEDIUM", "LOW"
comm_agent = CommunicationAgent(enable_mobile_alert=True)
comm_agent.handle_risk(risk_level)

 [ROAD DISPLAY SYSTEM]
 [CRITICAL] HIGH risk detected! Immediate action required.
 [MOBILE APP NOTIFICATION]
 [CRITICAL] HIGH risk detected! Immediate action required.


In [16]:
import datetime

class MonitoringAgent:
    def __init__(self, log_to_file=True, log_path="monitoring_log.txt"):
        self.log_to_file = log_to_file
        self.log_path = log_path

    def _write(self, text):
        timestamp = datetime.datetime.now().strftime("[%Y-%m-%d %H:%M:%S]")
        line = f"{timestamp} {text}"
        if self.log_to_file:
            with open(self.log_path, "a") as f:
                f.write(line + "\n")
        else:
            print(line)

    def log_detection_event(self, detections):
        self._write("📦 Detection Event:")
        for det in detections:
            self._write(f" - {det['class']} (conf={det['confidence']:.2f}) at {det['bbox']}")

    def log_risk_assessment(self, assessed):
        self._write("🧠 Risk Assessment:")
        for obj in assessed:
            self._write(f" - {obj['class']} → Risk = {obj['risk']}")

    def log_alert_status(self, risk_level):
        alert = "YES" if risk_level in ["HIGH", "MEDIUM"] else "NO"
        self._write(f"🚨 Alert Issued: {alert} (Overall Risk = {risk_level})")

final_risk = "HIGH"  


monitor = MonitoringAgent(log_to_file=True)  


monitor.log_detection_event(outputs)


monitor.log_risk_assessment(outputs)


monitor.log_alert_status(final_risk)

In [14]:
# Predict on test images
results = model.predict(source="/kaggle/working/animal_dataset/test/images", save=False, conf=0.25)
total_boxes = sum(len(r.boxes) for r in results)
print(f"\n Bounding boxes in the test images: {total_boxes}")



image 1/267 /kaggle/working/animal_dataset/test/images/00001_jpg.rf.a47ad67a15d9815ec15eaa4ebbb936ef.jpg: 448x640 2 1s, 30.8ms
image 2/267 /kaggle/working/animal_dataset/test/images/00006_jpg.rf.0629e1395e49c89fc4a97978d501578c.jpg: 448x640 2 1s, 30.7ms
image 3/267 /kaggle/working/animal_dataset/test/images/00007_jpg.rf.b88ae63fe80c1f8f27f91a1b632239c4.jpg: 448x640 1 1, 30.6ms
image 4/267 /kaggle/working/animal_dataset/test/images/00010_jpg.rf.a40a2f8fa36fcba2ee4a9d4e841796f5.jpg: 448x640 2 1s, 30.7ms
image 5/267 /kaggle/working/animal_dataset/test/images/00013_jpg.rf.546e5649a20eaf77a108ea479eba5aa6.jpg: 448x640 2 1s, 28.6ms
image 6/267 /kaggle/working/animal_dataset/test/images/00023_jpg.rf.2dde86b47027db39ac5bcac160366a48.jpg: 448x640 1 1, 28.4ms
image 7/267 /kaggle/working/animal_dataset/test/images/00032_jpg.rf.cfb6e2763aaf1e74f7efb1e76c119fee.jpg: 448x640 2 1s, 28.3ms
image 8/267 /kaggle/working/animal_dataset/test/images/00042_jpg.rf.18aded062f86cb99b66e08cae30694f4.jpg: 448x64

In [15]:
model = YOLO("/kaggle/working/runs/detect/animal_aug_train/weights/best.pt")  # hoặc yolov8.pt đã huấn luyện

# Dự đoán trên tập huấn luyện
results = model.predict(source="/kaggle/working/animal_dataset/train/images_aug", conf=0.25, save=False)

# Đếm tổng bounding box đã dự đoán
total_pred_boxes = sum(len(r.boxes) for r in results)

print(f"🔢 Tổng số bounding boxes YOLO đã dự đoán trên tập train: {total_pred_boxes}")
yolov8.pt


WARNING ⚠️ 
inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/5403 /kaggle/working/animal_dataset/train/images_aug/aug_0.jpg: 448x640 1 0, 30.9ms
image 2/5403 /kaggle/working/animal_dataset/train/images_aug/aug_1.jpg: 448x640 1 0, 28.3ms
image 3/5403 /kaggle/working/animal_dataset/train/images_aug/aug_10.jpg: 448x640 1 1, 28.3ms
image 4/5403 /kaggle/working/animal_dataset/train/images_aug/aug_100.jpg: 448x640 1 1, 28.2ms
image 5/5403 /kaggle/working/animal_dataset/train/images_aug/aug_1000.jpg: 448x640 1 

NameError: name 'yolov8' is not defined

In [17]:
from collections import Counter

box_class_counter = Counter()

for r in results:
    for c in r.boxes.cls:
        cls_id = int(c.item())
        box_class_counter[cls_id] += 1

print("\n📊 Số bounding box dự đoán theo lớp:")
for cls_id, count in box_class_counter.items():
    print(f"  Class {cls_id} ({model.names[cls_id]}): {count}")


📊 Số bounding box dự đoán theo lớp:
  Class 0 (0): 2996
  Class 1 (1): 4375
